In [2]:
from pyMoove import io, plotting, calculations, cleaning, statistics, displacement

from scripts.zone_info import get_zone_info
from pathlib import Path
import pandas as pd



In [ ]:

# Resolve data directory (works from project root or scripts directory)
DATA_DIR = Path("data")

data = io.load_processed_dataset(DATA_DIR)
zones_path = DATA_DIR / "zone_info.csv"
zones = get_zone_info(zones_path)

# Calculate statistics for all feeding bins
#bins = sorted(data["feeding"]["bin"].dropna().unique())
#bin_statistics = calculations.calculate_bin_position_statistics(data, bins, outlier_confidence=0.9999)
#bin_statistics.to_parquet(bin_statistics_path, index=False)

# Save results to Parquet
bin_statistics_path = DATA_DIR / "bin_position_stats.parquet"
bin_statistics = io.load_bin_stats_parquet(bin_statistics_path)

replacements = data['replacements']
stats = statistics.inspect_replacement_event_statistics(data['positions'], replacements, bin_statistics, time_before=30, time_after=30)
replacements = cleaning.filter_replacement_events(replacements, stats)
stats = statistics.inspect_replacement_event_statistics(data['positions'], replacements, bin_statistics, time_before=30, time_after=30)




Loaded dataset in folder: data
  positions Memory: 571.74 MiB
  feeding Memory: 0.28 MiB
  replacements Memory: 0.02 MiB
Total memory: 572.05 MiB

Total event before cleaning: 550
Events retained: 470
Expected removal: 80
Events removed: 80


In [34]:
# Looking at the results we might find outliers and would like to inspect these
#columns = ["cow_min_m"]
#columns = ["actor_bin_min_m"]
columns = ["reactor_bin_min_m"]

# If you want to select several outliers combine them uniqulie into one dataframe
# Otherwise find_boxplot_outliers() returns all outlier events for one specific column
outliers = []
for column in columns:
    outliers.append(statistics.find_boxplot_outliers(stats, column))

# All unique event outliers are collected into one dataframe.
unique_outliers = pd.concat(outliers).drop_duplicates(subset="event_id")

# We can inspect the outliers however we like
print(f"Total replacement events: {len(stats)}")
print(f"Unique outlier events: {len(unique_outliers)}")

print(unique_outliers.sort_values(columns, ascending=False).head(34)[column])


Total replacement events: 470
Unique outlier events: 30
345    8.797535
357    7.793606
150    6.337846
116    4.227981
279    4.212431
283    3.533455
251    3.130449
347    3.016121
286    2.145228
145    2.099333
228    2.083141
315    1.108620
334    0.988026
335    0.849140
338    0.777125
333    0.662339
82     0.648281
107    0.597914
423    0.589993
277    0.555043
76     0.554298
331    0.495799
332    0.466651
362    0.450121
56     0.428839
398    0.427224
461    0.422886
80     0.405398
350    0.394413
67     0.392254
Name: reactor_bin_min_m, dtype: float64


In [ ]:
print(f"Total replacement events: {len(replacements)}")
outliers_mask = (stats['actor_bin_min_m'] > 1) | (stats['reactor_bin_min_m'] > 1.11) | (stats['cow_min_m'] > 2.136)
replacements = replacements[~outliers_mask]
print(f"Total replacement events: {len(replacements)}")

In [ ]:
for bin_id in range(1, 21):
    bin_replacements = replacements.loc[replacements['bin']==bin_id]
    bin_replacements.sort_values('time').reset_index(drop=True)
    detected_replacements = displacement.detect_replacements_by_bin_hug(data['positions'], bin_statistics, bin_id, max_bin_distance=1.0)
    recall = displacement.check_replacement_recall(bin_replacements, detected_replacements, acceptance_interval=60, cow_role_matters=False, show_detections=False)

Actual replacements: 7
Detected replacements: 537
5 / 7 replacement events were detected
Actual replacements: 7
Detected replacements: 1116
6 / 7 replacement events were detected
Actual replacements: 10
Detected replacements: 1245
8 / 10 replacement events were detected
Actual replacements: 8
Detected replacements: 1561
7 / 8 replacement events were detected
Actual replacements: 9
Detected replacements: 1546
6 / 9 replacement events were detected
Actual replacements: 10
Detected replacements: 1330
6 / 10 replacement events were detected
Actual replacements: 3
Detected replacements: 1143
3 / 3 replacement events were detected
Actual replacements: 3
Detected replacements: 1189
2 / 3 replacement events were detected
Actual replacements: 4
Detected replacements: 981
3 / 4 replacement events were detected
Actual replacements: 9
Detected replacements: 1047
3 / 9 replacement events were detected
Actual replacements: 4
Detected replacements: 967
4 / 4 replacement events were detected
Actual re

In [35]:
animations = []
for i in [315]:
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:
animations = []
for i in range(0, 3):
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], bin_replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:

animations = []
for i in range(1265, 1270):
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], detected_replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:
bins = range(1,59)
plotting.plot_bins(zones, bin_statistics, bins)

In [ ]:
fig, ax = plotting.plot_feeding_event(data, zones, event_index=0, bin_statistics=bin_statistics)

In [ ]:

# Option A: Plot all feeding events for a specific bin (e.g. Bin 21, 24, etc.)
bins = range(58, 59)
for i in bins:
    target_bin = i
    fig, ax = plotting.plot_bin_feeding_positions(data, zones, bin_id=target_bin, color_by_cow=True, outlier_confidence=0.9999)

# Option B: Plot a single feeding event by index (uncomment to use):
# fig, ax = plot_feeding_event(data, zones, event_index=0)